# Lab 1 — 첫 LLM 호출 & 에이전트의 뼈대

> **이론 복습 — Session 1 슬라이드**
> - LLM은 "다음 토큰 예측기"다. **텍스트만** 생성한다.
> - LLM의 3가지 한계: ① 환각 ② 멈춰 있는 지식 ③ 행동 불가
> - 에이전트 = LLM + `Tools` + `Memory` + `Loop` + `Instructions`
>
> 이번 실습에서는 에이전트의 가장 안쪽 부품 — **LLM 호출** 부터 시작합니다.
> 도구(`Tools`)는 다음 시간(S2), 루프(`Loop`)는 S3에서 붙입니다.

## 학습 목표
1. `common/llm.py` 로 LLM을 호출한다
2. LLM의 한계 ①②③ 을 **직접 재현**해 본다
3. 시스템 프롬프트로 출력을 **통제**한다
4. 도구 없이, 프롬프트만으로 "추론 → 행동 제안" 형식을 끌어낸다 (ReAct 맛보기)


## 0. 준비

- `pip install -r requirements.txt` 가 끝나 있어야 합니다.
- `.env` 파일에 `GEMINI_API_KEY` 를 넣으면 **실제 모델**로 동작합니다.
  무료 키 발급: <https://aistudio.google.com/apikey>
- **키가 없어도 괜찮습니다** — 자동으로 `MockLLM`(가짜 LLM)이 대신 동작해
  실습의 모든 단계가 끝까지 실행됩니다. (단, 답변 내용은 가짜입니다.)
- 이 노트북은 `labs/` 폴더에서 실행하세요.


In [ ]:
# `common/` 패키지를 불러옵니다.
from common.llm import LLMClient

# API 키가 있으면 Gemini, 없으면 MockLLM 으로 자동 연결됩니다.
llm = LLMClient()
print("mock 모드인가요?", llm.is_mock)

## 1. 첫 LLM 호출

가장 단순한 사용법입니다. 문자열을 넣으면 `LLMResponse` 가 돌아옵니다.
`.text` 에 모델의 답변이 들어 있습니다.


In [ ]:
reply = llm.generate("Agentic AI를 학부생에게 한 문장으로 설명해줘.")
print(reply.text)

## 2. LLM의 한계를 직접 재현하기

슬라이드에서 본 3가지 한계를 코드로 확인합니다.
**키가 있다면** 모델의 진짜 반응을, 어떻게 "그럴듯하게" 빗나가는지 관찰하세요.


In [ ]:
# 한계 ② — 멈춰 있는 지식: 모델은 '지금'을 알 수 없다.
print(llm.generate("지금 현재 시각을 초 단위까지 정확히 알려줘.").text)

In [ ]:
# 한계 ① — 환각: 존재하지 않는 사실도 그럴듯하게 지어낸다.
print(llm.generate("DTUMOS 프로젝트가 2026년에 받은 상의 정확한 명칭을 알려줘.").text)

In [ ]:
# 한계 ③ — 행동 불가: 모델은 우리 데이터 파일을 열어볼 수 없다.
print(llm.generate(
    "labs/data/od_flows.json 파일에서 통행량이 가장 큰 구간을 알려줘."
).text)

**관찰 포인트**

- 한계 ②③ 의 답은 "알 수 없다"가 아니라, 보통 *그럴듯한 추측*으로 나옵니다.
- 한계 ① 은 아예 없는 사실을 만들어내기도 합니다.

> 바로 이 세 한계를 메우는 것이 **도구(S2)** 와 **에이전트 루프(S3)** 입니다.
> 예: `od_flows.json` 을 읽는 *도구*를 주면, 한계 ③ 이 해결됩니다.


## 3. 시스템 프롬프트로 출력 통제하기

에이전트의 `Instructions` 요소입니다. 같은 질문이라도 **시스템 프롬프트**로
역할·말투·형식을 지정하면 답이 달라집니다.


In [ ]:
# (a) 시스템 프롬프트 없이
print("--- 시스템 프롬프트 없음 ---")
print(llm.generate("교통 혼잡이 뭐야?").text)

In [ ]:
# (b) 시스템 프롬프트로 역할과 형식을 지정
system = (
    "You are a transportation expert teaching undergraduate students. "
    "Always answer in Korean, in exactly 2 sentences, "
    "and include one everyday example."
)
print("--- 시스템 프롬프트 있음 ---")
print(llm.generate("교통 혼잡이 뭐야?", system=system).text)

### few-shot — 예시로 형식을 가르치기

프롬프트 안에 **입력→출력 예시**를 몇 개 보여주면, 모델은 그 형식을 따라 합니다.


In [ ]:
prompt = '''다음 형식으로 분류해줘.

입력: 강남에서 판교로 가는 통근
출력: {"종류": "광역통근", "추천수단": "광역버스"}

입력: 종로에서 시청으로 가는 통근
출력: {"종류": "도심통근", "추천수단": "지하철"}

입력: 수원에서 서울역으로 가는 통근
출력:'''
print(llm.generate(prompt).text)

## 4. 도구 없이 "추론 → 행동" 끌어내기 (ReAct 맛보기)

진짜 도구 호출은 다음 시간(S2)입니다. 지금은 시스템 프롬프트만으로
모델이 **생각(Thought)** 과 **행동 제안(Action)** 을 *글로* 쓰게 만들어 봅니다.

이것이 S3에서 구현할 **추론→행동→관찰 루프**의 씨앗입니다.


In [ ]:
react_system = '''You are a mobility analysis agent.
You cannot run tools yet, so instead THINK step by step and PROPOSE an action.
Reply in exactly this format, in Korean:

Thought: <무엇을 알아내야 하는지 한 줄>
Action: <호출하면 좋을 가상의 도구와 입력. 예: get_top_flows(n=5)>
'''

question = "통근 통행량이 가장 많은 상위 5개 구간이 궁금해."
print(llm.generate(question, system=react_system).text)

모델이 "어떤 도구를, 어떤 입력으로 부르면 좋을지"를 스스로 말합니다.
S2에서는 이 *제안*을 **진짜 함수 호출**로 바꿉니다.


## 🔧 TODO — 나만의 시스템 프롬프트 작성

아래 `my_system` 에 **모빌리티 분석가** 역할의 시스템 프롬프트를 작성하세요.

요구사항:
- 한국어로 답한다
- 답은 **3문장 이내**
- 항상 끝에 "추가로 확인하면 좋을 분석" 한 가지를 제안한다


In [ ]:
# ✅ 해답 예시 — 정답은 하나가 아닙니다. 요구사항만 충족하면 됩니다.
my_system = """You are a mobility analyst for Korean cities.
Rules:
- Always answer in Korean.
- Keep the answer to 3 sentences or fewer.
- End every answer with one concrete suggestion for a follow-up analysis,
  prefixed with "추가 분석 제안:".
"""

# --- 테스트 ---
test_q = "서울과 경기 사이 통근 통행이 많은 이유가 뭘까?"
result = llm.generate(test_q, system=my_system)
print(result.text)

## 정리

- LLM은 텍스트 생성기다 — 한계 ①②③ 을 직접 확인했다.
- 시스템 프롬프트(`Instructions`)로 역할·형식을 통제할 수 있다.
- 프롬프트만으로도 "추론→행동" 형식을 끌어낼 수 있다.

**다음 — Session 2 (도구 사용)**
> 모델의 "행동 제안"을 *진짜 함수 호출*로 바꾸는 **function calling** 을 배웁니다.
> 그러면 한계 ③(행동 불가)이 사라집니다.
